## Prepare dataset in the required format

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import re
import os

input_file = "/content/drive/My Drive/master_thesis_project/uzh_metadata_compact.json"
output_file = "/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only.jsonl"

# GitHub raw URL prefix
github_prefix = "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(output_file, "w", encoding="utf-8") as f_out:
    for obj in data:
        material_raw = obj.get("medium", "")
        material = material_raw.split(",")[0].strip() if material_raw else "" # due to "ton,gebrannt"

        culture_raw = obj.get("culture", "") # Culture: keep only the first entry if list, otherwise clean string
        if isinstance(culture_raw, list):
            kultur = culture_raw[0].strip() if culture_raw else ""
        else:
            kultur = str(culture_raw).strip()

        kategorie = obj.get("category", "")

        images = obj.get("images", [])
        if not images:
            continue

        # Replace local paths with GitHub URLs
        github_images = []
        for img_path in images[:10]: # ensure max 10 images
            filename = os.path.basename(img_path)  # extract e.g., "4121_image_0.jpg"
            github_images.append({"type": "image_url", "image_url": {"url": github_prefix + filename}})

        # build conversation - SINGLE TURN FORMAT
        messages = [
            {
                "role": "system",
                "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."
            },
            {
                "role": "user",
                "content": github_images + [
                    {"type": "text", "text": "Bitte gib die Metadaten (Material, Kultur, Kategorie) für dieses Artefakt an."}
                ]
            },
            {
                "role": "assistant",
                "content": f"Material: {material}\nKultur: {kultur}\nKategorie: {kategorie}"
            }
        ]

        entry = {"messages": messages}
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nDone! JSONL with GitHub image URLs written to {output_file}")


Done! JSONL with GitHub image URLs written to /content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only.jsonl


In [ ]:
# Show the first 5 lines of the JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

{"messages": [{"role": "system", "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_0.jpg"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_1.jpg"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_2.jpg"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_3.jpg"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_4.jpg"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanita

In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 0:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_0.jpg"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_1.jpg"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_2.jpg"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh

Large images would be more tokens and cost more. According to https://platform.openai.com/docs/guides/vision-fine-tuning#size :
If you set the detail parameter for an image to low, the image is resized to 512 by 512 pixels and is only represented by 85 tokens regardless of its size. This will reduce the cost of training.

In [ ]:
import os
from PIL import Image

# Path to your image folder
image_folder = "/content/drive/MyDrive/master_thesis_project/uzh_images"

# List all files in that folder
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith('.jpg')]

too_large = []

for file_name in image_files:
    image_path = os.path.join(image_folder, file_name)
    try:
        with Image.open(image_path) as img:
            width, height = img.size
            if width > 512 or height > 512:
                too_large.append((file_name, width, height))
    except Exception as e:
        print(f"Could not open {file_name}: {e}")

# Show results
if too_large:
    print(f"{len(too_large)} out of {len(image_files)} images are larger than 512px:")
    for name, w, h in too_large:
        print(f" - {name}: {w}x{h}px")
else:
    print("All images are 512px or smaller. Perfect!")

4675 out of 4675 images are larger than 512px:
 - 2240_image_3.jpg: 914x1200px
 - 2240_image_4.jpg: 914x1200px
 - 2240_image_5.jpg: 914x1200px
 - 2240_image_6.jpg: 1200x951px
 - 2240_image_7.jpg: 1200x964px
 - 2240_image_8.jpg: 1200x1193px
 - 2240_image_9.jpg: 1200x1051px
 - 2240_image_10.jpg: 1200x1051px
 - 2240_image_11.jpg: 1200x916px
 - 2240_image_12.jpg: 1200x904px
 - 2240_image_13.jpg: 778x1200px
 - 2240_image_14.jpg: 949x1200px
 - 2240_image_15.jpg: 778x1200px
 - 2240_image_16.jpg: 802x1200px
 - 2240_image_17.jpg: 778x1200px
 - 2240_image_18.jpg: 778x1200px
 - 2240_image_19.jpg: 778x1200px
 - 2240_image_20.jpg: 778x1200px
 - 2240_image_21.jpg: 778x1200px
 - 2240_image_22.jpg: 1108x1200px
 - 2240_image_23.jpg: 1108x1200px
 - 2240_image_24.jpg: 718x1200px
 - 2240_image_25.jpg: 718x1200px
 - 2240_image_26.jpg: 927x1200px
 - 2240_image_27.jpg: 940x1200px
 - 2240_image_28.jpg: 1034x1200px
 - 2240_image_29.jpg: 1043x1200px
 - 2240_image_30.jpg: 1160x1200px
 - 2240_image_31.jpg: 1143x1

In [ ]:
import os
import json

jsonl_file = "/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only.jsonl"

# Check file size
file_size_mb = os.path.getsize(jsonl_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Check number of images per example
max_images_allowed = 10
supported_formats = (".jpg", ".jpeg", ".png", ".webp")
max_10_images = True

with open(jsonl_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            example = json.loads(line)
            user_content = example["messages"][1]["content"]
            if not isinstance(user_content, list):
                print(f"Example {i}: user content is not a list")
                continue

            # Extract image URLs
            image_urls = [x["image_url"]["url"] for x in user_content if x.get("type") == "image_url"]

            if len(image_urls) > max_images_allowed:
                print(f"Example {i}: has {len(image_urls)} images (consider reducing to {max_images_allowed})")
                max_10_images = False

            # check formats
            for url in image_urls:
                if not url.lower().endswith(supported_formats):
                    print(f"Example {i}: unsupported image format -> {url}")

        except Exception as e:
            print(f"Error processing example {i}: {e}")

if max_10_images:
    print(f"All the artefacts have maximal 10 images.")

File size: 0.70 MB
All the artefacts have maximal 10 images.


https://community.openai.com/t/gpt-4-vision-preview-fidelity-detail-parameter/477563

add the parameter detail=low into data  

e.g. taken from https://platform.openai.com/docs/guides/vision-fine-tuning#size  

{
  "type": "image_url",
  "image_url": {
    "url": "https://upload.wikimedia.org/wikipedia/commons/3/36/Danbo_Cheese.jpg",
    "detail": "low"
  }
}

In [ ]:
import json

# Input and output paths
input_file = "/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only.jsonl"
output_file = "/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only_detail_low.jsonl"

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        data = json.loads(line)

        # Go through messages
        for message in data.get("messages", []):
            if isinstance(message.get("content"), list):
                for item in message["content"]:
                    if (isinstance(item, dict) and
                        item.get("type") == "image_url" and
                        "image_url" in item):
                        item["image_url"]["detail"] = "low"

        # Write updated data
        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print("All image entries updated with detail='low' and saved to:")
print(output_file)

All image entries updated with detail='low' and saved to:
/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only_detail_low.jsonl


In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 0:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_0.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_1.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6213_image_2.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "im

### Split the data into train, val and test set in the ratio of 8:1:1

In [ ]:
import json
import random
from pathlib import Path

input_file = "/content/drive/My Drive/master_thesis_project/uzh_data_for_gpt4o_classification_only_detail_low.jsonl"
output_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Load all examples
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

total = len(lines)
print(f"Total examples: {total}")

# Shuffle the data
random.shuffle(lines)

# Compute split indices
train_end_idx = int(total * 0.8)
val_end_idx = int(total * 0.9)

train = lines[:train_end_idx]
val = lines[train_end_idx:val_end_idx]
test = lines[val_end_idx:]

# Save splits
splits = {"train": train, "validation": val, "test": test}

for split_name, split_data in splits.items():
    output_file = Path(output_dir) / f"{split_name}.jsonl"
    with open(output_file, "w", encoding="utf-8") as f_out:
        for line in split_data:
            f_out.write(line)
    print(f"{split_name}: {len(split_data)} examples saved to {output_file}")

# Count total images for each split file
print("\n=== Image Counts ===")
for split_name in ["train", "validation", "test"]:
    split_file = Path(output_dir) / f"{split_name}.jsonl"
    total_images = 0

    with open(split_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            for message in data.get("messages", []):
                content = message.get("content")
                if isinstance(content, list):
                    for item in content:
                        if isinstance(item, dict) and item.get("type") == "image_url":
                            total_images += 1

    print(f"{split_name}: {total_images} total images")

Total examples: 495
train: 396 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/train.jsonl
validation: 49 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/validation.jsonl
test: 50 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/test.jsonl

=== Image Counts ===
train: 2946 total images
validation: 382 total images
test: 386 total images


In [ ]:
# make sure the encoding is utf-8 for gpt4o fine-tuning
import chardet

with open(output_file, "rb") as f:
    raw = f.read(4096)
    print(chardet.detect(raw))

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


### Fine-tuning

In [ ]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/948.4 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.107.0
    Uninstalling openai-1.107.0:
      Successfully uninstalled openai-1.107.0


#### load the inspect the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
uzh_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only"

In [ ]:
!wc -l "{uzh_data_dir}/train.jsonl"
!wc -l "{uzh_data_dir}/validation.jsonl"
!wc -l "{uzh_data_dir}/test.jsonl"

396 /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/train.jsonl
49 /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/validation.jsonl
50 /content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only/test.jsonl


In [ ]:
!head -n 10 "{uzh_data_dir}/train.jsonl"

{"messages": [{"role": "system", "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_0.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_1.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_2.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_3.jpg", "detail": "low"}}, {"type": "image_url", "image_url": {"url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_4.jpg", "detail": "low"}}, 

In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(f"{uzh_data_dir}/train.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 0:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "Du bist eine Museumskuratorin, die präzise und fachlich korrekte Metadaten archäologischer Artefakte auf Deutsch erstellt."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_0.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_1.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/6433_image_2.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "im

#### run gpt4o fine-tuning

use the fine-tunable vision model gpt-4o-2024-08-06  
https://platform.openai.com/docs/guides/vision-fine-tuning . But in this link, it says that "Each example can have at most 10 images."

Accoding to https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/fine-tuning-vision :
Vision fine-tuning is supported for gpt-4o version 2024-08-06 and gpt-4.1 version 2025-04-14 models only. In this link, it says "Each example can have at most 64 images." This is written in 2025. but in this link https://platform.openai.com/docs/guides/supervised-fine-tuning gpt-4.1 version 2025-04-14 models are not listed in the vision fine-tuning page.

In [ ]:
# initiate openai client

from openai import OpenAI

client = OpenAI(api_key="sk-xxx")

upload the train and validation files

In [ ]:
training_file_upload_response = client.files.create(
    file=open(f"{uzh_data_dir}/train.jsonl", "rb"),
    purpose="fine-tune"
)

validation_file_upload_response = client.files.create(
    file=open(f"{uzh_data_dir}/validation.jsonl", "rb"),
    purpose="fine-tune"
)

print("training_file_upload_response:", training_file_upload_response)
print("validation_file_upload_response:", validation_file_upload_response)

training_file_upload_response: FileObject(id='file-F5F4KcNNfNLGbckLDyamac', bytes=631053, created_at=1761303169, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
validation_file_upload_response: FileObject(id='file-ShsAz5a6bVAra6zvxffv7k', bytes=80856, created_at=1761303170, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


create a fine-tuned mode

In [ ]:
from datetime import datetime
import re

def generate_finetune_suffix(project_name: str, dataset_name: str, version: str = None) -> str:
    """
    Generates a kebab-case suffix for fine-tuning jobs, where spaces are replaced with hyphens and all letters are lower-case.

    Args:
        project_name (str): Short name for the project, e.g., 'artefacts'.
        dataset_name (str): Name of the dataset, e.g., 'uzh'.
        version (str, optional): Version or date, e.g., 'v1' or '2025-10-15'. Defaults to today.

    Returns:
        str: kebab-case suffix, e.g., 'artefacts-uzh-2025-10-15'
    """
    def to_kebab(text: str) -> str:
        return re.sub(r'\s+', '-', text.strip().lower())

    if version is None:
        version = datetime.today().strftime("%Y-%m-%d")

    parts = [project_name, dataset_name, version]

    return '-'.join(to_kebab(part) for part in parts)


fine_tuning_response = client.fine_tuning.jobs.create(
    training_file=training_file_upload_response.id,
    validation_file=validation_file_upload_response.id,
    suffix=generate_finetune_suffix('artefacts', 'uzh', 'v6'),
    model="gpt-4o-2024-08-06"

)

fine_tuning_response

FineTuningJob(id='ftjob-0muacgRq62KUFwCWRjOxjeLK', created_at=1761303190, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=630301335, status='validating_files', trained_tokens=None, training_file='file-F5F4KcNNfNLGbckLDyamac', validation_file='file-ShsAz5a6bVAra6zvxffv7k', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='artefacts-uzh-v6', usage_metrics=None, shared_with_openai=False, eval_id=None)

check training job status

In [ ]:
status_response = client.fine_tuning.jobs.retrieve(fine_tuning_response.id)

status_response

FineTuningJob(id='ftjob-0muacgRq62KUFwCWRjOxjeLK', created_at=1761303190, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=630301335, status='running', trained_tokens=None, training_file='file-F5F4KcNNfNLGbckLDyamac', validation_file='file-ShsAz5a6bVAra6zvxffv7k', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix='artefacts-uzh-v6', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="sk-xxx")
status = client.fine_tuning.jobs.retrieve('ftjob-0muacgRq62KUFwCWRjOxjeLK')
status.status

'succeeded'

run inference using fine-tuned model

In [ ]:
from openai import OpenAI

#uzh_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/uzh_split_data/classification_only"

client = OpenAI(api_key="sk-xxx")
status_response = client.fine_tuning.jobs.retrieve('ftjob-0muacgRq62KUFwCWRjOxjeLK')
fine_tuned_model = status_response.fine_tuned_model

In [ ]:
fine_tuned_model

'ft:gpt-4o-2024-08-06:university-of-zurich-department-of-history:artefacts-uzh-v6:CUAhNQLk'

In [ ]:
import json
from openai import OpenAI
import re

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl(f"{uzh_data_dir}/test.jsonl")

completion = client.chat.completions.create(
    model=status_response.fine_tuned_model,
    messages=test_data[0]['messages'][:-1]
)

completion.choices[0].message

ChatCompletionMessage(content='Material: Ton\nKultur: Attisch\nKategorie: Gefäss', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [ ]:
# Quick check of training data format
# Load the training data first
with open(f"{uzh_data_dir}/train.jsonl", "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

# Now check the first 5 samples
for i in range(5):
    assistant_content = train_data[i]['messages'][2]['content']
    print(f"Sample {i}:")
    print(assistant_content[:200] + "..." if len(assistant_content) > 200 else assistant_content)
    print("---")

Sample 0:
Material: Ton
Kultur: Kampanisch
Kategorie: Gefäss
---
Sample 1:
Material: Ton
Kultur: Apulisch
Kategorie: Gefäss
---
Sample 2:
Material: Ton
Kultur: Apulisch
Kategorie: Gefäss
---
Sample 3:
Material: Ton
Kultur: Sizilisch
Kategorie: Gefäss
---
Sample 4:
Material: Metall
Kultur: Subgeometrisch
Kategorie: Kleinplastik
---


In [ ]:
import json
from openai import OpenAI
import re
from tqdm import tqdm

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Helper to parse metadata into a dict
def parse_metadata(text):
    """
    Converts text like:
    'Material: Ton\nKultur: Korinthisch\nKategorie: Gefäss'
    into {'Material': 'Ton', 'Kultur': 'Korinthisch', 'Kategorie': 'Gefäss'}
    """
    meta = {}
    for line in text.split("\n"):
        if ':' in line:
            key, val = line.split(":", 1)
            meta[key.strip()] = val.strip()
    return meta

def generate_metadata_predictions(data, model, max_samples=None):
    predictions = []

     # Calculate total_samples
    total_samples = len(data) if max_samples is None else min(max_samples, len(data))

    for i, sample in enumerate(tqdm(data, total=total_samples, desc="Generating metadata")):
        if max_samples and i >= max_samples:
            break

        messages = sample["messages"]

        # Ground truths
        ground_metadata = messages[2]["content"]  # assistant metadata

        # First image URL
        first_image_url = None
        for c in messages[1]["content"]:
            if c.get("type") == "image_url":
                first_image_url = c["image_url"]["url"]
                break

        # Predict metadata
        conversation = [
            messages[0],  # system
            messages[1],  # user asking for metadata
        ]

        completion = client.chat.completions.create(
            model=model,
            messages=conversation
        )
        prediction_metadata = completion.choices[0].message.content.strip()

        # Compare metadata fields
        gt_meta_dict = parse_metadata(ground_metadata)
        pred_meta_dict = parse_metadata(prediction_metadata)
        metadata_match = []
        for key in ["Material", "Kultur", "Kategorie"]:
            match = "✅" if gt_meta_dict.get(key, "").lower() == pred_meta_dict.get(key, "").lower() else "❌"
            metadata_match.append(f"{key} {match}")
        metadata_match_str = ", ".join(metadata_match)

        # Save predictions
        predictions.append({
            "sample_id": i + 1,
            "first_image_url": first_image_url,
            "ground_metadata": ground_metadata,
            "prediction_metadata": prediction_metadata,
            "metadata_match": metadata_match_str,
        })

    return predictions

In [ ]:
# Load test set

test_data = load_jsonl(f"{uzh_data_dir}/test.jsonl")

# Generate predictions
test_preds = generate_metadata_predictions(test_data, fine_tuned_model)

# Print nicely
def print_predictions(preds):
    for r in preds:
        print(json.dumps(r, indent=2, ensure_ascii=False))

print("\n=== Test Set Predictions ===")
print_predictions(test_preds)

Generating metadata: 100%|██████████| 50/50 [04:10<00:00,  5.01s/it]


=== Test Set Predictions ===
{
  "sample_id": 1,
  "first_image_url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/20065_image_0.jpg",
  "ground_metadata": "Material: Ton\nKultur: Attisch\nKategorie: Gefäss",
  "prediction_metadata": "Material: Ton\nKultur: Attisch\nKategorie: Gefäss",
  "metadata_match": "Material ✅, Kultur ✅, Kategorie ✅"
}
{
  "sample_id": 2,
  "first_image_url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/2791_image_0.jpg",
  "ground_metadata": "Material: Ton\nKultur: Attisch\nKategorie: Gefäss",
  "prediction_metadata": "Material: Ton\nKultur: Apulisch\nKategorie: Gefäss",
  "metadata_match": "Material ✅, Kultur ❌, Kategorie ✅"
}
{
  "sample_id": 3,
  "first_image_url": "https://raw.githubusercontent.com/juanitalfr5-pixel/uzh_artefacts_images/main/4602_image_0.jpg",
  "ground_metadata": "Material: Ton\nKultur: Magna Graecia\nKategorie: Gefäss",
  "prediction_metadata": "Material: Ton\nKultur: Ap

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

# Metadata metrics
def compute_metadata_metrics(predictions):
    gt_material, gt_kultur, gt_kategorie = [], [], []
    pred_material, pred_kultur, pred_kategorie = [], [], []

    for p in predictions:
        # parse ground truth
        gt = parse_metadata(p['ground_metadata'])
        pred = parse_metadata(p['prediction_metadata'])

        gt_material.append(gt.get("Material", "").lower())
        gt_kultur.append(gt.get("Kultur", "").lower())
        gt_kategorie.append(gt.get("Kategorie", "").lower())

        pred_material.append(pred.get("Material", "").lower())
        pred_kultur.append(pred.get("Kultur", "").lower())
        pred_kategorie.append(pred.get("Kategorie", "").lower())

    # Compute accuracy, precision, recall, f1 for each field
    def compute_metrics(gt_list, pred_list, label):
        acc = accuracy_score(gt_list, pred_list)
        prec = precision_score(gt_list, pred_list, average='macro', zero_division=0)
        rec = recall_score(gt_list, pred_list, average='macro', zero_division=0)
        f1 = f1_score(gt_list, pred_list, average='macro', zero_division=0)
        print(f"--- {label} ---\nAccuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}\n")

    compute_metrics(gt_material, pred_material, "Material")
    compute_metrics(gt_kultur, pred_kultur, "Kultur")
    compute_metrics(gt_kategorie, pred_kategorie, "Kategorie")

# Run metrics
compute_metadata_metrics(test_preds)

--- Material ---
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000

--- Kultur ---
Accuracy: 0.7000, Precision: 0.3323, Recall: 0.3938, F1: 0.3557

--- Kategorie ---
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000

